In [1]:
import os
import librosa
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPClassifier

from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier

# =====================================================
# DATASET
# =====================================================

DATASET = "/media/feliciano/Aux/AI_AFS_DATASET/behavior_dataset"

CLASSES = [
    "normal",
    "clustering",
    "agitation",
    "other"
]

SR = 16000

# =====================================================
# FEATURE EXTRACTION
# =====================================================

X = []
y = []

for label in CLASSES:

    folder = os.path.join(DATASET, label)

    files = [
        f for f in os.listdir(folder)
        if f.endswith(".wav")
    ]

    print(f"{label}: {len(files)} files")

    for file in files:

        path = os.path.join(folder, file)

        try:

            signal, sr = librosa.load(
                path,
                sr=SR,
                mono=True
            )

            signal = (
                signal - np.mean(signal)
            ) / (
                np.std(signal) + 1e-8
            )

            features = []

            # ==========================================
            # RMS
            # ==========================================

            features.append(
                np.mean(
                    librosa.feature.rms(
                        y=signal
                    )
                )
            )

            # ==========================================
            # ZCR
            # ==========================================

            features.append(
                np.mean(
                    librosa.feature.zero_crossing_rate(
                        signal
                    )
                )
            )

            # ==========================================
            # Spectral Features
            # ==========================================

            features.append(
                np.mean(
                    librosa.feature.spectral_centroid(
                        y=signal,
                        sr=sr
                    )
                )
            )

            features.append(
                np.mean(
                    librosa.feature.spectral_bandwidth(
                        y=signal,
                        sr=sr
                    )
                )
            )

            features.append(
                np.mean(
                    librosa.feature.spectral_rolloff(
                        y=signal,
                        sr=sr
                    )
                )
            )

            # ==========================================
            # Spectral Contrast
            # ==========================================

            contrast = librosa.feature.spectral_contrast(
                y=signal,
                sr=sr
            )

            features.extend(
                np.mean(
                    contrast,
                    axis=1
                )
            )

            # ==========================================
            # LOG MEL FEATURES
            # ==========================================

            mel = librosa.feature.melspectrogram(
                y=signal,
                sr=sr,
                n_mels=64
            )

            log_mel = librosa.power_to_db(
                mel,
                ref=np.max
            )

            features.extend(
                np.mean(
                    log_mel,
                    axis=1
                )
            )

            X.append(features)
            y.append(label)

        except Exception as e:

            print(
                "ERROR:",
                path,
                e
            )

# =====================================================
# NUMPY
# =====================================================

X = np.array(
    X,
    dtype=np.float32
)

y = np.array(y)

print("\nSamples:", len(X))
print("Features:", X.shape[1])

# =====================================================
# ENCODE LABELS
# =====================================================

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

print("\nClasses")

for i, cls in enumerate(
    encoder.classes_
):
    print(i, cls)

# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    stratify=y_encoded,
    random_state=42
)

# =====================================================
# MODELS
# =====================================================

models = {

    "Logistic Regression": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]),

    "Gaussian NB": GaussianNB(),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=300,
        learning_rate=0.5,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softmax",
        num_class=len(CLASSES),
        eval_metric="mlogloss",
        random_state=42
    ),

    "KNN": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            KNeighborsClassifier(
                n_neighbors=7
            )
        )
    ]),

    "SVM (RBF)": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            SVC(
                kernel="rbf",
                C=10,
                gamma="scale"
            )
        )
    ]),

    "MLP": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            MLPClassifier(
                hidden_layer_sizes=(128, 64),
                learning_rate_init=0.001,
                max_iter=300,
                random_state=42
            )
        )
    ])
}

# =====================================================
# TRAIN / EVALUATE
# =====================================================

results = []

for name, model in models.items():

    print("\n")
    print("=" * 80)
    print(name)
    print("=" * 80)

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_test
    )

    acc = accuracy_score(
        y_test,
        pred
    )

    precision, recall, f1, _ = \
        precision_recall_fscore_support(
            y_test,
            pred,
            average="macro"
        )

    results.append(
        {
            "Model": name,
            "Accuracy": round(acc, 4),
            "Precision": round(precision, 4),
            "Recall": round(recall, 4),
            "F1": round(f1, 4)
        }
    )

    print(
        f"\nAccuracy : {acc:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"Recall   : {recall:.4f}"
    )

    print(
        f"F1 Score : {f1:.4f}"
    )

    print("\nClassification Report\n")

    print(
        classification_report(
            y_test,
            pred,
            target_names=encoder.classes_
        )
    )

    print("\nConfusion Matrix\n")

    print(
        confusion_matrix(
            y_test,
            pred
        )
    )

# =====================================================
# COMPARISON TABLE
# =====================================================

results_df = pd.DataFrame(
    results
)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\n")
print("=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

print(results_df)

results_df.to_csv(
    "salmon_model_comparison.csv",
    index=False
)

print(
    "\nSaved: salmon_model_comparison.csv"
)

normal: 8049 files
clustering: 1260 files
agitation: 1140 files
other: 240 files

Samples: 10689
Features: 76

Classes
0 agitation
1 clustering
2 normal
3 other


Logistic Regression

Accuracy : 0.7947
Precision: 0.6264
Recall   : 0.8471
F1 Score : 0.6911

Classification Report

              precision    recall  f1-score   support

   agitation       0.44      0.86      0.58       228
  clustering       0.68      0.87      0.76       252
      normal       0.98      0.77      0.86      1610
       other       0.40      0.90      0.55        48

    accuracy                           0.79      2138
   macro avg       0.63      0.85      0.69      2138
weighted avg       0.87      0.79      0.81      2138


Confusion Matrix

[[ 195    7   24    2]
 [   5  218    4   25]
 [ 241   89 1243   37]
 [   0    5    0   43]]


Gaussian NB

Accuracy : 0.6305
Precision: 0.4709
Recall   : 0.6988
F1 Score : 0.5014

Classification Report

              precision    recall  f1-score   support

   agit